# Day 3 — Aggregation & Visualization

Goal: Answer simple questions with pandas and create two plots.

Tasks: Top items by revenue, revenue by date (time series). Questions will be defined after day 2.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set(style='whitegrid')

csv_path = Path('sales_clean.csv')
if not csv_path.exists():
    # create a small sample CSV so the notebook can run without the real data
    sample = [
        {'scraped_at': '2026-06-30T12:00:00', 'item': 'Croissant', 'price': 2.5, 'quantity': 1, 'total': 2.5, 'store': 'Main', 'url': 'u1'},
        {'scraped_at': '2026-06-30T12:05:00', 'item': 'Baguette', 'price': 3.0, 'quantity': 2, 'total': 6.0, 'store': 'Main', 'url': 'u2'},
        {'scraped_at': '2026-07-01T09:00:00', 'item': 'Croissant', 'price': 2.5, 'quantity': 3, 'total': 7.5, 'store': 'Branch', 'url': 'u3'}
    ]
    pd.DataFrame(sample).to_csv(csv_path, index=False, encoding='utf-8')
    print('Wrote sample sales_clean.csv - replace with your real file')

# Read the CSV, parse scraped_at as datetime if present
df = pd.read_csv(csv_path, parse_dates=['scraped_at'], encoding='utf-8')
print('Loaded rows:', len(df))
print('Column types:')
print(df.dtypes)


## Top items by revenue
Group by `item` and sum `total`. Save top 5 as a bar chart (horizontal).

In [ ]:
# Ensure total column exists; if not, compute it
if 'total' not in df.columns:
    if 'price' in df.columns and 'quantity' in df.columns:
        df['total'] = df['price'] * df['quantity']
    else:
        df['total'] = 0

top_items = df.groupby('item', as_index=False)['total'].sum().sort_values('total', ascending=False)
top5 = top_items.head(5)
print('Top 5 items by revenue:')
print(top5.to_string(index=False))

# Plot horizontal bar chart
plt.figure(figsize=(8,5))
sns.barplot(data=top5, x='total', y='item', palette='viridis')
plt.title('Top 5 items by revenue')
plt.xlabel('Revenue')
plt.tight_layout()
plt.savefig('top5_items.png')
print('Saved plot: top5_items.png')
plt.show()


## Daily revenue time series
Aggregate by date and plot a line chart.

In [ ]:
# Normalize scraped_at to date if present, else try to parse a date column
if 'scraped_at' in df.columns:
    df['date'] = pd.to_datetime(df['scraped_at'], errors='coerce').dt.date
elif 'date' in df.columns:
    df['date'] = pd.to_datetime(df['date'], errors='coerce').dt.date
else:
    df['date'] = pd.to_datetime('2026-06-30').date()

daily = df.groupby('date', as_index=False)['total'].sum().sort_values('date')
print('Daily revenue sample:')
print(daily.head().to_string(index=False))

# Plot time series
plt.figure(figsize=(10,4))
plt.plot(pd.to_datetime(daily['date']), daily['total'], marker='o')
plt.title('Daily revenue')
plt.xlabel('Date')
plt.ylabel('Revenue')
plt.tight_layout()
plt.savefig('daily_revenue.png')
print('Saved plot: daily_revenue.png')
plt.show()
